[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diogoflim/AM/blob/main/03_naive_bayes.ipynb)


# Aula: Naive Bayes

Nesta aula, combinaremos:

- **intuição probabilística**;
- **implementação prática com `scikit-learn`**;
- **comparações entre diferentes versões do método**.



## Objetivos de aprendizagem

Ao final deste notebook, espera-se que você seja capaz de:

1. interpretar a ideia de classificação probabilística;
2. entender o papel do **Teorema de Bayes**;
3. compreender a hipótese de independência condicional do Naive Bayes;
4. treinar e avaliar modelos com a biblioteca `scikit-learn`;
5. implementar partes importantes do fluxo de modelagem;
6. comparar variantes como:
   - `GaussianNB`
   - `MultinomialNB`
   - `BernoulliNB`

---



## Ideia central

Em classificação, queremos prever o rótulo $y^{(i)}$ de um novo exemplo $x^{(i)}$.

Uma regra natural é escolher a classe mais provável dado o vetor de atributos observado:


$\hat{y} = \arg\max_{c \in \mathcal{Y}} P(Y=c \mid X=x)$

Esse critério é chamado de **MAP** (*Maximum A Posteriori*).

Pelo Teorema de Bayes:


$P(Y=c \mid X=x) = \frac{P(X=x \mid Y=c)\,P(Y=c)}{P(X=x)}$

Como $P(X=x)$ é igual para todas as classes durante a comparação, obtemos:

$\hat{y} = \arg\max_{c \in \mathcal{Y}} P(X=x \mid Y=c)\,P(Y=c)$


Assim, para classificar, precisamos estimar:

- a probabilidade **a priori** da classe: $P(Y=c)$;
- a probabilidade de observar os atributos dado a classe: $P(X=x \mid Y=c)$.


## A hipótese "naive"

Se $\vec{x}=(x_1,\dots,x_p)$, estimar diretamente $P(X=\vec{x} \mid Y=c)$ pode ser muito difícil, especialmente quando há muitos atributos.

O Naive Bayes faz a hipótese de que os atributos são **independentes condicionalmente à classe**:

$$
P(X_1=x_1,\dots,X_p=x_p \mid Y=c) = \prod_{j=1}^{p} P(X_j = x_j \mid Y=c)
$$

Então a regra de decisão fica:

$$
\hat{y} = \arg\max_{c \in \mathcal{Y}} P(Y=c)\prod_{j=1}^{p} P(X_j=x_j \mid Y=c)
$$

Essa hipótese geralmente não é exatamente verdadeira, mas o método costuma funcionar muito bem na prática.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#pd.set_option("display.max_columns", None)


## Primeiro exemplo: Iris

Vamos começar com o conjunto de dados **Iris**, que possui:

- 150 observações;
- 4 atributos numéricos contínuos;
- 3 classes.

Como os atributos são contínuos, a variante mais natural é o **Gaussian Naive Bayes**.


In [ ]:
from sklearn.datasets import load_iris, load_wine, load_digits

In [ ]:
iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="classe")

X.head()

In [ ]:
y.value_counts().sort_index()


## Separação em treino e teste


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, Binarizer

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Tamanho de treino:", X_train.shape)
print("Tamanho de teste :", X_test.shape)

## Treinando o Gaussian Naive Bayes

No `scikit-learn`, o fluxo básico é:

1. criar o modelo;
2. ajustar com `fit`;
3. obter previsões com `predict`;
4. avaliar o desempenho.


In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)

y_pred = gnb.predict(X_test)

print("Acurácia:", round(accuracy_score(y_test, y_pred), 4))


## Relatório de classificação


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
print(classification_report(y_test, y_pred, target_names=iris.target_names))


### Como interpretar esse relatório?

- **precision**: entre os exemplos previstos como pertencentes a uma classe, qual proporção realmente pertence a ela;
- **recall**: entre os exemplos que realmente pertencem a uma classe, qual proporção foi corretamente identificada;
- **f1-score**: média harmônica entre precision e recall;
- **support**: quantidade de exemplos reais de cada classe no conjunto de teste.

A **accuracy** é a proporção total de acertos.

Como o conjunto Iris é relativamente equilibrado, as médias `macro avg` e `weighted avg` tendem a ser próximas.


## Matriz de confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris.target_names)
disp.plot()
plt.show()


A matriz de confusão permite observar **quais classes estão sendo confundidas entre si**.

Nos problemas multiclasses, ela é uma ferramenta importante porque a acurácia sozinha não mostra *onde* o modelo está errando.


## O que o Gaussian Naive Bayes assume?

No caso gaussiano, para cada classe $c$ e atributo $X_j$, supõe-se que:

$X_j \mid Y=c \sim \mathcal{N}(\mu_{jc}, \sigma^2_{jc})$

Ou seja, cada atributo contínuo é modelado por uma distribuição normal dentro de cada classe.

Logo, o algoritmo precisa estimar:

- as probabilidades a priori $P(Y=c)$;
- a média $\mu_{jc}$ de cada atributo em cada classe;
- a variância $\sigma^2_{jc}$ de cada atributo em cada classe.



## 10. Inspecionando os parâmetros aprendidos


In [ ]:
priors = pd.Series(gnb.class_prior_, index=iris.target_names, name="prior")
priors

In [ ]:
medias = pd.DataFrame(gnb.theta_, columns=iris.feature_names, index=iris.target_names)
medias

In [ ]:
variancias = pd.DataFrame(gnb.var_, columns=iris.feature_names, index=iris.target_names)
variancias


Essas tabelas mostram exatamente o que o modelo estimou a partir dos dados de treinamento.

Em outras palavras, o `GaussianNB` constrói um modelo probabilístico simples para cada classe.



# Exercícios práticos

A partir daqui, os exercícios foram pensados para que os alunos **programem**, testem funções e utilizem a biblioteca na prática.

Sempre que possível, evite responder apenas com texto: a ideia é **codificar**.



## Exercício 1 — Treino e avaliação em outro conjunto de dados

Repita o fluxo abaixo para o dataset **Wine**:

1. carregue o dataset `load_wine()`;
2. faça a separação treino/teste com `test_size=0.25`, `random_state=42` e `stratify=y`;
3. treine um `GaussianNB`;
4. calcule a acurácia;
5. exiba a matriz de confusão.

**Objetivo:** praticar o uso da biblioteca em outro problema de classificação multiclasses.


In [ ]:
# EXERCÍCIO 1
# Escreva sua solução nesta célula.

# Dica:
# from sklearn.datasets import load_wine



## Exercício 2 — Função de avaliação

Implemente uma função chamada `avaliar_modelo_nb` que receba:

- um modelo do `scikit-learn`;
- `X_train`, `X_test`, `y_train`, `y_test`.

A função deve:

1. ajustar o modelo;
2. prever os rótulos do conjunto de teste;
3. retornar um dicionário com:
   - `"accuracy"`
   - `"y_pred"`

Depois, use essa função para avaliar um `GaussianNB` no dataset Iris.

**Objetivo:** treinar encapsulamento de código e fluxo de modelagem.


In [ ]:
# EXERCÍCIO 2
# Implemente a função abaixo.

def avaliar_modelo_nb(modelo, X_train, X_test, y_train, y_test):
    pass



## Exercício 3 — Comparando train/test split diferentes

Usando o dataset Iris, compare o desempenho do `GaussianNB` para os seguintes valores de `test_size`:

- 0.20
- 0.25
- 0.30
- 0.40

Armazene os resultados em um `DataFrame` com as colunas:

- `test_size`
- `accuracy`

**Objetivo:** perceber que a avaliação depende da divisão dos dados.


In [ ]:
# EXERCÍCIO 3
# Monte um loop e gere um DataFrame com os resultados.



## Exercício 4 — Probabilidades previstas

Treine um `GaussianNB` no Iris e utilize o método:

```python
predict_proba(X_test)
```

Depois:

1. transforme a saída em um `DataFrame`;
2. nomeie as colunas com os nomes das espécies;
3. mostre as 10 primeiras linhas.

**Objetivo:** interpretar o modelo de forma probabilística, e não apenas pelas classes finais.


In [ ]:
# EXERCÍCIO 4
# Use predict_proba e organize a saída em um DataFrame.



## 11. Entendendo `predict_proba`

Vamos ver como o modelo retorna probabilidades a posteriori estimadas.


In [ ]:
probs = gnb.predict_proba(X_test)
df_probs = pd.DataFrame(probs, columns=iris.target_names)
df_probs.head(10)


Cada linha representa um exemplo do conjunto de teste, e cada coluna indica a probabilidade estimada de pertencimento a uma classe.

A previsão final (`predict`) corresponde à classe com maior valor em cada linha.



## 12. Comparando `predict` e `predict_proba`


In [ ]:
comparacao = X_test.copy().reset_index(drop=True)
comparacao["classe_real"] = y_test.reset_index(drop=True).map(dict(enumerate(iris.target_names)))
comparacao["classe_prevista"] = pd.Series(y_pred).map(dict(enumerate(iris.target_names)))

for nome in iris.target_names:
    comparacao[f"prob_{nome}"] = df_probs[nome].values

comparacao.head(10)


## Exercício 5 — Casos mais incertos

Usando a tabela acima ou o resultado de `predict_proba`, identifique os **5 exemplos mais incertos** do conjunto de teste.

Sugestão: uma forma de medir incerteza é usar a maior probabilidade prevista em cada linha. Quanto menor esse valor máximo, maior a incerteza do modelo.

Crie uma tabela com:

- índice do exemplo;
- classe real;
- classe prevista;
- maior probabilidade prevista.

**Objetivo:** praticar manipulação de probabilidades e interpretação do classificador.


In [ ]:
# EXERCÍCIO 5
# Gere a tabela solicitada.



## 13. Variantes do Naive Bayes

O `scikit-learn` oferece diferentes variantes do Naive Bayes, adequadas a diferentes tipos de atributos:

### GaussianNB
Usado quando os atributos são contínuos e podem ser aproximados por distribuições normais.

### MultinomialNB
Muito utilizado com contagens, por exemplo:
- frequência de palavras em documentos;
- número de ocorrências de eventos.

### BernoulliNB
Usado quando os atributos são binários:
- presença/ausência;
- sim/não;
- 0/1.



## 14. Exemplo com MultinomialNB

Para ilustrar o `MultinomialNB`, vamos usar o dataset `digits` e transformar os atributos para valores não negativos em uma escala mais conveniente.

O importante aqui não é a perfeição da modelagem estatística, mas sim **experimentar a biblioteca**.


In [ ]:
digits = load_digits()

X_digits = pd.DataFrame(digits.data)
y_digits = pd.Series(digits.target)

X_digits.min().min(), X_digits.max().max()

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 16))
X_digits_scaled = pd.DataFrame(scaler.fit_transform(X_digits)).round()

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_digits_scaled, y_digits, test_size=0.25, random_state=42, stratify=y_digits
)

mnb = MultinomialNB()
mnb.fit(X_train_d, y_train_d)
y_pred_d = mnb.predict(X_test_d)

print("Acurácia do MultinomialNB:", round(accuracy_score(y_test_d, y_pred_d), 4))


## Exercício 6 — Comparação entre GaussianNB e MultinomialNB

No dataset `digits`, treine e compare:

- `GaussianNB`
- `MultinomialNB`

Monte um `DataFrame` com duas linhas e as colunas:

- `modelo`
- `accuracy`

**Objetivo:** praticar comparação entre modelos.


In [ ]:
# EXERCÍCIO 6
# Compare os dois modelos no dataset digits.



## 15. Exemplo com BernoulliNB

Agora vamos binarizar os pixels do dataset `digits`: cada atributo passará a indicar presença/ausência de intensidade acima de um limiar.

Isso cria um cenário compatível com `BernoulliNB`.


In [ ]:
binarizador = Binarizer(threshold=8.0)
X_digits_bin = pd.DataFrame(binarizador.fit_transform(X_digits))

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_digits_bin, y_digits, test_size=0.25, random_state=42, stratify=y_digits
)

bnb = BernoulliNB()
bnb.fit(X_train_b, y_train_b)
y_pred_b = bnb.predict(X_test_b)

print("Acurácia do BernoulliNB:", round(accuracy_score(y_test_b, y_pred_b), 4))


## Exercício 7 — Investigando o limiar de binarização

Repita o experimento do `BernoulliNB` no dataset `digits`, alterando o valor de `threshold` do `Binarizer`.

Teste pelo menos os valores:

- 4
- 8
- 12

Monte um `DataFrame` com as colunas:

- `threshold`
- `accuracy`

**Objetivo:** perceber que decisões de pré-processamento afetam o desempenho do modelo.


In [ ]:
# EXERCÍCIO 7
# Monte o experimento solicitado.



## 16. Por que trabalhar com log-probabilidades?

Na regra do Naive Bayes, multiplicamos várias probabilidades:

\[
P(Y=c)\prod_{j=1}^{p} P(X_j=x_j \mid Y=c)
\]

Como probabilidades são números entre 0 e 1, o produto de muitos termos pode ficar extremamente pequeno.

Por isso, costuma-se trabalhar com logaritmos:

\[
\log P(Y=c) + \sum_{j=1}^{p} \log P(X_j=x_j \mid Y=c)
\]

Essa transformação:

- evita problemas numéricos;
- preserva a ordem de comparação entre as classes.



## Exercício 8 — Produto de probabilidades vs soma de log-probabilidades

Crie um vetor com várias probabilidades pequenas, por exemplo:

```python
p = np.array([0.12, 0.08, 0.15, 0.20, 0.05, 0.11, 0.09, 0.07])
```

Depois:

1. calcule o produto direto dessas probabilidades;
2. calcule a soma dos logaritmos;
3. verifique que:

```python
np.exp(np.sum(np.log(p)))
```

recupera o mesmo valor do produto direto (salvo pequenas diferenças numéricas).

**Objetivo:** praticar uma ideia central da implementação do Naive Bayes.


In [ ]:
# EXERCÍCIO 8
# Faça os cálculos pedidos.



## 17. Exercício integrador

Escolha **um** dos datasets usados neste notebook:

- `iris`
- `wine`
- `digits`

e construa um pequeno experimento completo.

Seu código deve:

1. carregar os dados;
2. separar treino e teste;
3. escolher uma variante adequada do Naive Bayes;
4. treinar o modelo;
5. calcular a acurácia;
6. exibir a matriz de confusão;
7. mostrar pelo menos 5 probabilidades previstas com `predict_proba`.

**Objetivo:** consolidar o pipeline completo de uso da biblioteca.


In [ ]:
# EXERCÍCIO 9
# Resolva aqui o experimento integrador.



## 18. Vantagens e limitações

### Vantagens
- simples de implementar;
- rápido para treinar;
- rápido para prever;
- funciona bem em muitos problemas reais;
- pode ser competitivo mesmo com a hipótese de independência violada.

### Limitações
- a hipótese de independência pode ser forte demais;
- atributos redundantes podem distorcer a evidência;
- a qualidade da modelagem depende do tipo de atributo;
- algumas variantes exigem pré-processamento apropriado.



## 19. Fechamento conceitual

Na aula anterior, com **k-NN**, a ideia era classificar usando a vizinhança local dos exemplos.

Agora, com **Naive Bayes**, classificamos usando um **modelo probabilístico**:

\[
\hat{y} = \arg\max_{c} P(Y=c)\prod_j P(X_j \mid Y=c)
\]

Portanto, ambos são classificadores, mas a lógica interna é diferente:

- **k-NN**: baseado em proximidade;
- **Naive Bayes**: baseado em probabilidade.

Essa comparação é importante porque mostra que diferentes algoritmos podem atacar o mesmo problema de maneiras bastante distintas.



## 20. Desafio opcional

Implemente uma função chamada `comparar_naive_bayes` que receba:

- `X_train`, `X_test`, `y_train`, `y_test`

e devolva um `DataFrame` comparando as acurácias de:

- `GaussianNB`
- `MultinomialNB` (quando aplicável)
- `BernoulliNB` (quando aplicável)

Você pode adaptar os dados conforme necessário.

---
Fim do notebook.
